# NER Analysis — BERT-base Token Classification Results
Analyze NER training curves, per-entity F1, and prediction examples.

In [ ]:
import sys
sys.path.insert(0, '..')
import json
import matplotlib.pyplot as plt
from pathlib import Path

results_path = Path('../results/ner_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print('NER Evaluation Results:')
    print(f"  Overall F1:        {results['overall_f1']}")
    print(f"  Overall Precision: {results['overall_precision']}")
    print(f"  Overall Recall:    {results['overall_recall']}")
    print('\nPer-entity:')
    for entity, metrics in results['per_entity'].items():
        print(f"  {entity:12s}  P={metrics['precision']:.3f}  R={metrics['recall']:.3f}  F1={metrics['f1']:.3f}")
else:
    print('Run the NER training stage first.')

In [ ]:
# Training curves
log_path = Path('../results/ner_training_log.jsonl')
if log_path.exists():
    epochs, train_losses, val_losses, val_f1s = [], [], [], []
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line)
            epochs.append(entry['epoch'])
            train_losses.append(entry['train_loss'])
            val_losses.append(entry['val_loss'])
            val_f1s.append(entry['val_f1'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_losses, label='Train', marker='o')
    axes[0].plot(epochs, val_losses, label='Val', marker='o')
    axes[0].set_title('NER Training and Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(epochs, val_f1s, color='green', marker='o')
    axes[1].set_title('Validation F1 per Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1')
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-entity F1 bar chart
if results_path.exists():
    entities = list(results['per_entity'].keys())
    f1s = [results['per_entity'][e]['f1'] for e in entities]
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(entities, f1s, color=colors[:len(entities)])
    ax.axhline(results['overall_f1'], color='red', linestyle='--', label=f"Overall F1={results['overall_f1']}")
    ax.set_title('NER F1 by Entity Type')
    ax.set_ylabel('F1')
    ax.set_ylim(0, 1.0)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Semi-supervised expansion results
ss_path = Path('../results/semisup_expansion_log.json')
if ss_path.exists():
    with open(ss_path) as f:
        ss = json.load(f)
    print(f"Total examples added: {ss['total_examples_added']}")
    print(f"Final val F1: {ss['final_val_f1']}")
    for it in ss['iterations']:
        print(f"  Iter {it['iteration']}: +{it['examples_added']} examples, F1 {it['val_f1_before']} -> {it['val_f1_after']}")